# 09.2 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/02-sentiment-classifier/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1: by hand

With the toy votes *great* +2, *good* +1, *not* −2, *bad* −2: score *"good good bad"* and *"not not bad"*. Then add the bigram *not bad* with vote +5 (count it **as well as** the single words) and rescore *"not bad"*. Check with code.

In [ ]:
votes = {"great": 2, "good": 1, "not": -2, "bad": -2}

def toy_score(review):
    words = review.lower().replace(",", "").split()
    z = sum(votes.get(w, 0) for w in words)
    return z, 1 / (1 + np.exp(-z))

# your code here

## Exercise 2: `min_df`

Train the bigram model with `min_df` of 1, 2, 5 and 20. Print vocabulary size and test accuracy.

In [ ]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def text_model(**vectoriser_settings):
    return make_pipeline(TfidfVectorizer(**vectoriser_settings), LogisticRegression(max_iter=1000))

model = text_model(ngram_range=(1, 2), min_df=2).fit(train["text"], train["label"])

# your code here

## Exercise 3: fool the model

Write five reviews of ShopWave products (mugs, fairy lights, cake stands…) that you think the model will get wrong. How many does it?

In [ ]:
# your code here

## Exercise 4: learning curve

Train the bigram model on the first 100, 500, 2,000 and 8,000 training reviews and print test accuracy. Keep these numbers for 09.3.

In [ ]:
# your code here